In [6]:
from ngo.ml.basisfunctions import *
from ngo.ml.quadrature import *

In [43]:
N = 200
n_t = 2
n_x = 10
n_y = 10
Q_t = 3
Q_x = 99
Q_y = 99

n_elements_t = 1
n_elements_x = 3
n_elements_y = 3

basis_t = BSplineBasis1D(h=n_t, p=1, C=0)
basis_x = BSplineBasis1D(h=n_x, p=3, C=2)
basis_y = BSplineBasis1D(h=n_y, p=3, C=2)

quadrature_t = GaussLegendreQuadrature(Q=[Q_t], n_elements=[n_elements_t])
quadrature_x = GaussLegendreQuadrature(Q=[Q_x], n_elements=[n_elements_x])
quadrature_y = GaussLegendreQuadrature(Q=[Q_y], n_elements=[n_elements_y])
t = quadrature_t.xi.reshape(quadrature_t.xi.shape[0],)
x = quadrature_x.xi.reshape(quadrature_x.xi.shape[0],) 
y = quadrature_y.xi.reshape(quadrature_y.xi.shape[0],)

X = t,x,y

quadrature = GaussLegendreQuadrature(Q=(Q_t,Q_x,Q_y), n_elements=(n_elements_t,n_elements_x,n_elements_y))
X_q = quadrature.xi

u_m = np.random.uniform(size=N)
U = u_m.reshape((n_t, n_x, n_y))


In [51]:
class TensorProductBasis:
    """
    A class to represent tensorized basis functions.
    This class combines multiple 1D basis functions into a higher-dimensional tensorized/Kronecker product basis.

    Attributes:
        bases (tuple): A tuple of 1D basis objects.
        d (int): Dimensionality of the tensorized basis (number of 1D bases).
        n_basisfunctions (int): Total number of basis functions in the tensorized basis.
    """
    def __init__(self, bases):
        """
        Initialize the tensorized basis.

        Args:
            bases (list): A list of 1D basis objects.
        """
        self.bases = bases
        self.d = len(bases)
        self.h = ()
        for i in range(len(bases)):
            self.h += (bases[i].h,)
    
    def forward(self, u_m, x):
        """
        Compute the tensorized basis values at given points.

        Args:
            x (np.ndarray): Tuple of input points (t, x, y, ...) where t, x, y have lengths n_t, n_x, n_y.

        Returns:
            np.ndarray: Tensorized basis values at the input points. Shape: (n_points, n_basisfunctions).
        """
        # u_m = 
        u_q = u_m.reshape(self.h)
        for i in range(self.d):
           Phi_i = self.bases[i].forward(x[i])
           u_q = np.tensordot(u_q, Phi_i.T, axes=(0,0))
           u_q = u_q
        return u_q
    
    def grad(self, x):
        """
        Compute the gradients of the tensorized basis functions.

        Args:
            x (np.ndarray): Input points. Shape: (n_points, d).
            
        Returns:
            np.ndarray: Gradients of the tensorized basis functions at the input points.
                        Shape: (n_points, n_basisfunctions, d).
        """
        tensorproduct = np.zeros((x.shape[0], self.n_basisfunctions, self.d))
        for i in range(self.d):
            if i==0:
                tensorproduct_i = self.bases[0].grad(x[:,0])
            else:
                tensorproduct_i = self.bases[0].forward(x[:,0])
            newsize = self.bases[0].h
            for j in range(1,self.d):
                newsize = newsize*self.bases[j].h
                if j==i:
                    tensorproduct_i = opt_einsum.contract('Nm,Nn->Nmn', tensorproduct_i, self.bases[j].grad(x[:,j])).reshape(x.shape[0], newsize)
                else: 
                    tensorproduct_i = opt_einsum.contract('Nm,Nn->Nmn', tensorproduct_i, self.bases[j].forward(x[:,j])).reshape(x.shape[0], newsize)
            tensorproduct[:,:,i] = tensorproduct_i
        return tensorproduct


In [52]:
basis1 = TensorProductBasis(bases=(basis_t,basis_x,basis_y))

In [53]:
result1 = basis1.forward(u_m, X)#.reshape((Q_t*Q_x*Q_y))

In [54]:
basis2 = TensorizedBasis(bases=[basis_t,basis_x,basis_y])

In [55]:
Phi2 = basis2.forward(X_q)

In [56]:
result2 = np.einsum('n,qn->q', u_m, Phi2).reshape(Q_t,Q_x,Q_y)

In [57]:
result2 - result1

array([[[-5.55111512e-17,  0.00000000e+00, -1.11022302e-16, ...,
          1.11022302e-16,  2.22044605e-16,  0.00000000e+00],
        [ 5.55111512e-17,  0.00000000e+00,  0.00000000e+00, ...,
         -3.33066907e-16,  0.00000000e+00,  2.22044605e-16],
        [ 1.11022302e-16,  0.00000000e+00,  0.00000000e+00, ...,
         -1.11022302e-16,  1.11022302e-16, -1.11022302e-16],
        ...,
        [-5.55111512e-17, -5.55111512e-17,  0.00000000e+00, ...,
          0.00000000e+00,  5.55111512e-17,  1.11022302e-16],
        [ 5.55111512e-17,  0.00000000e+00, -1.11022302e-16, ...,
          5.55111512e-17, -5.55111512e-17,  0.00000000e+00],
        [-1.11022302e-16, -5.55111512e-17, -1.11022302e-16, ...,
          5.55111512e-17, -5.55111512e-17, -5.55111512e-17]],

       [[ 0.00000000e+00, -1.11022302e-16, -5.55111512e-17, ...,
          1.11022302e-16,  0.00000000e+00, -1.11022302e-16],
        [ 0.00000000e+00,  0.00000000e+00, -5.55111512e-17, ...,
          0.00000000e+00,  2.22044605e